# Clase 3 · Laboratorio — ETL completo: Saber 11 → Bodega en SQLite

**Trabajo en parejas · 90 min.**

En la pre-clase construyeron `dim_colegio` y un hecho parcial. Hoy completan la bodega:
- Agregar `dim_tiempo` y `dim_geografia`
- Actualizar `hecho_resultados` con las 3 claves de referencia
- Cargar el modelo estrella completo a SQLite
- Ejecutar 3 consultas analíticas reales

**Checkpoint del profesor a los 50 min** (después de la Tarea 3).

In [ ]:
import pandas as pd
import sqlite3

CSV = "../../datos/saber11_muestra_500k.csv"
DB  = "../../datos/saber11_lab_etl.db"

df_raw = pd.read_csv(CSV, dtype=str)
df = df_raw.copy()

# Transformaciones base
cols_puntaje = ["PUNT_LECTURA_CRITICA","PUNT_MATEMATICAS","PUNT_C_NATURALES",
                "PUNT_SOCIALES_CIUDADANAS","PUNT_INGLES","PUNT_GLOBAL"]
for col in cols_puntaje:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["COLE_NATURALEZA","COLE_JORNADA","COLE_CALENDARIO","COLE_BILINGUE",
            "COLE_DEPTO_UBICACION","COLE_MCPIO_UBICACION"]:
    df[col] = df[col].str.strip().str.upper()

df["PERIODO"] = pd.to_numeric(df["PERIODO"], errors="coerce").astype("Int64")

print(f"Datos listos: {len(df):,} filas × {df.shape[1]} columnas")

## Tarea 1 (10 min) — Reconstruir dim_colegio + añadir dim_tiempo y dim_geografia

Ya construyeron `dim_colegio` en la pre-clase. Aquí la reconstruyen rápido y añaden las dos dimensiones que faltan.

| Dimensión | Columnas (además del ID) |
|---|---|
| `dim_colegio` | COLE_NATURALEZA, COLE_JORNADA, COLE_CALENDARIO, COLE_BILINGUE |
| `dim_tiempo` | PERIODO |
| `dim_geografia` | COLE_DEPTO_UBICACION, COLE_MCPIO_UBICACION |

In [ ]:
# dim_colegio (igual que en pre-clase)
dim_colegio = (
    df[["COLE_NATURALEZA","COLE_JORNADA","COLE_CALENDARIO","COLE_BILINGUE"]]
    .drop_duplicates().reset_index(drop=True)
)
dim_colegio.insert(0, "colegio_id", dim_colegio.index + 1)
assert dim_colegio["colegio_id"].is_unique

# TODO: dim_tiempo — partir PERIODO en anio y quarter
# Ejemplo: 20254 → anio=2025, quarter=4
# Pista: anio = PERIODO // 10   |   quarter = PERIODO % 10
dim_tiempo = ...
# Debe tener columnas: tiempo_id, PERIODO, anio, quarter
assert dim_tiempo["tiempo_id"].is_unique

# TODO: dim_geografia con geo_id (COLE_DEPTO_UBICACION + COLE_MCPIO_UBICACION)
dim_geografia = ...
assert dim_geografia["geo_id"].is_unique

print(f"dim_colegio: {len(dim_colegio)} | dim_tiempo: {len(dim_tiempo)} | dim_geografia: {len(dim_geografia)}")
print("\ndim_tiempo:")
print(dim_tiempo)

## Tarea 2 (20 min) — hecho_resultados con las 3 claves de referencia

En la pre-clase el hecho solo tenía `colegio_id`. Ahora hay que agregar `tiempo_id` y `geo_id`.

1. Une `df` con las 3 dimensiones mediante `.merge(..., how='left')`.
2. El hecho final debe tener: `colegio_id`, `tiempo_id`, `geo_id` + los 6 puntajes.
3. **Validación:** `len(hecho_resultados) == len(df)` — si falla, hay un join mal configurado.

In [ ]:
# Unir con las 3 dimensiones para obtener las claves de referencia
df_h = df.merge(dim_colegio,
                on=["COLE_NATURALEZA","COLE_JORNADA","COLE_CALENDARIO","COLE_BILINGUE"],
                how="left")

# TODO: merge con dim_tiempo (on="PERIODO")
df_h = df_h.merge(...)

# TODO: merge con dim_geografia (on=["COLE_DEPTO_UBICACION","COLE_MCPIO_UBICACION"])
df_h = df_h.merge(...)

cols_puntaje = ["PUNT_LECTURA_CRITICA","PUNT_MATEMATICAS","PUNT_C_NATURALES",
                "PUNT_SOCIALES_CIUDADANAS","PUNT_INGLES","PUNT_GLOBAL"]
hecho_resultados = df_h[["colegio_id","tiempo_id","geo_id"] + cols_puntaje].copy()

assert len(hecho_resultados) == len(df), f"Perdimos {len(df)-len(hecho_resultados)} filas"
print(f"hecho_resultados: {len(hecho_resultados):,} filas con 3 FKs ✓")
hecho_resultados.head(3)

## Tarea 3 (15 min) — Cargar el modelo estrella completo a SQLite

Carga las 4 tablas a la BD. Luego verifica contando las filas de cada tabla.

In [ ]:
conn = sqlite3.connect(DB)

# TODO: cargar las 4 tablas con to_sql (if_exists='replace')
# dim_colegio, dim_tiempo, dim_geografia, hecho_resultados

# Verificar
for tabla in ["dim_colegio", "dim_tiempo", "dim_geografia", "hecho_resultados"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
    print(f"  {tabla}: {n:,} filas")

## ⏸ Checkpoint del profesor (10 min)

Revisamos juntos:
- ¿Alguien perdió filas en la Tarea 2? Diagnóstico rápido.
- ¿Por qué las bodegas de datos NO imponen restricciones de FK en la base de datos?
- ¿Qué representa una fila del hecho_resultados? ¿Un estudiante, un colegio, un período?

## Tarea 4 (30 min) — Diagrama del modelo estrella + Consultas SQL

### Parte A — Diagrama del modelo (5 min)

Aquí está el modelo que construyeron. Añadan en el Markdown de abajo los atributos de cada tabla:

```
            dim_tiempo
           (tiempo_id, PERIODO)
                  ↑
dim_colegio ← hecho_resultados → dim_geografia
(colegio_id,  (colegio_id (ref.),    (geo_id,
NATURALEZA,    tiempo_id (ref.),      DEPTO,
JORNADA,       geo_id (ref.),         MUNICIPIO)
CALENDARIO,    PUNT_LC,
BILINGUE)      PUNT_MAT, ...)
```

In [ ]:
# Consulta 1: promedio de PUNT_GLOBAL por naturaleza del colegio (oficial vs no oficial)
q1 = """
SELECT c.COLE_NATURALEZA,
       ROUND(AVG(h.PUNT_GLOBAL), 1) AS prom_global,
       COUNT(*) AS n_estudiantes
FROM hecho_resultados h
JOIN dim_colegio c ON h.colegio_id = c.colegio_id
GROUP BY c.COLE_NATURALEZA
ORDER BY prom_global DESC
"""
r1 = pd.read_sql(q1, conn)
print("Consulta 1:")
print(r1)

In [ ]:
# Consulta 2: top 5 departamentos con mayor promedio de PUNT_MATEMATICAS
q2 = """
-- TODO: escribe la consulta
-- Pista: necesitas JOIN con dim_geografia
"""
r2 = pd.read_sql(q2, conn)
print("Consulta 2:")
print(r2)

In [ ]:
# Consulta 3: por año y quarter, ¿en qué período mejoró más el puntaje?
q3 = """
-- TODO: escribe la consulta
-- Muestra el promedio de PUNT_GLOBAL agrupado por t.anio y t.quarter
-- Ordena cronológicamente (anio ASC, quarter ASC)
-- Pista: JOIN con dim_tiempo usando tiempo_id
"""
r3 = pd.read_sql(q3, conn)
print("Consulta 3:")
print(r3)

conn.close()

### Conclusión

Mirando los resultados de las 3 consultas, escribe 3-4 frases:
- ¿Qué brecha de puntaje hay entre colegios oficiales y no oficiales?
- ¿La jornada con mejor puntaje coincide con lo que esperabas?

---

## Reflexión final — ¿Qué dimensión faltó?

Revisa las columnas del CSV de Saber 11 que **no usaste** en ninguna dimensión:

```
ESTU_GENERO, ESTU_FECHANACIMIENTO
FAMI_ESTRATOVIVIENDA, FAMI_TIENEINTERNET
FAMI_EDUCACIONMADRE, FAMI_EDUCACIONPADRE
DESEMP_INGLES
```

**Preguntas:**
1. ¿A qué dimensión pertenecen estas columnas? ¿Cómo la llamarías?
2. ¿Cuáles columnas incluirías en esa dimensión y cuáles dejarías fuera? ¿Por qué?
3. Bosqueja el código para construirla (solo la estructura — no es necesario ejecutarla):

```python
# dim_??? = (
#     df[["...", "...", "..."]]
#     .drop_duplicates()
#     .reset_index(drop=True)
# )
# dim_???.insert(0, "???_id", dim_???.index + 1)
```

_Tus respuestas:_

## Entrega

- Suban este notebook completado a Moodle antes de las 23:59.
- Nombre: `apellido1_apellido2_clase03_lab.ipynb`.